# Lesson 5 - MCP CDC Agent Gateway

A real MCP client discovers and calls a local stdio server using the [official Python SDK](https://py.sdk.modelcontextprotocol.io/v1/). CDC updates reach the read-only tools through an exercise-owned SQLite database. No API key is needed.

> Synthetic training data only; not for clinical use.

### Setup

From the repository root: `pip install -r exercises/shared/requirements.txt`.

Keep this lesson and `shared` as sibling folders. Each example starts and stops its MCP process; the final cell removes the temporary database.

In [1]:
import pathlib
import sys

_SHARED = next(
    (candidate / "shared" for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
     for candidate in (parent, parent / "exercises")
     if (candidate / "shared" / "cdc_lessons").is_dir()),
    None,
)
if _SHARED is None:
    raise RuntimeError("Keep the shared folder alongside the Lesson 4 and 5 folders.")
if str(_SHARED) not in sys.path:
    sys.path.insert(0, str(_SHARED))

import json
from uuid import uuid4
from datetime import UTC, datetime, timedelta

from cdc_lessons.agent import DeterministicMCPClient
from cdc_lessons.mcp_client import call_tool, run_mcp_example
from cdc_lessons.runtime import create_lesson_db, update_referral

database = pathlib.Path.cwd() / f".lesson05-{uuid4().hex}.db"
db = create_lesson_db(f"sqlite:///{database.as_posix()}")
print(f"Exercise database: {database.name}")

Exercise database: .lesson05-fb2c64a1a0ea429dbd24cceab7f0392c.db


## (a) Discover MCP tools and entities

`ClientSession.list_tools()` reads schemas from the server. Only five read tools are registered.

In [2]:
async def discover(session):
    tools = (await session.list_tools()).tools
    assert {tool.name for tool in tools} == {
        "list_entities", "get_record", "search_records",
        "get_change_history", "get_freshness_status",
    }
    for tool in tools:
        print(tool.name, json.dumps(tool.inputSchema))
        assert tool.annotations.readOnlyHint

    entities = await call_tool(session, "list_entities")
    print("Entities:", [item["entity"] for item in entities["data"]])
    assert {item["entity"] for item in entities["data"]} == {
        "care_work_queue", "patient_timeline"
    }

run_mcp_example(database, discover)

list_entities {"properties": {}, "title": "list_entitiesArguments", "type": "object"}
get_record {"properties": {"entity": {"title": "Entity", "type": "string"}, "record_id": {"title": "Record Id", "type": "string"}, "fields": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Fields"}}, "required": ["entity", "record_id"], "title": "get_recordArguments", "type": "object"}
search_records {"properties": {"entity": {"title": "Entity", "type": "string"}, "min_priority_score": {"anyOf": [{"type": "integer"}, {"type": "null"}], "default": null, "title": "Min Priority Score"}, "limit": {"default": 5, "title": "Limit", "type": "integer"}}, "required": ["entity"], "title": "search_recordsArguments", "type": "object"}
get_change_history {"properties": {"entity": {"title": "Entity", "type": "string"}, "record_id": {"title": "Record Id", "type": "string"}, "limit": {"default": 10, "title": "Limit", "type": "integer"}}, "required": ["entity", "

## (b) Read through MCP

Tool responses contain structured data and evidence IDs. Datetimes cross the protocol as ISO strings.

In [3]:
async def read_baseline(session):
    record = await call_tool(session, "get_record", entity="care_work_queue", record_id="p4")
    history = await call_tool(session, "get_change_history", entity="care_work_queue", record_id="p4")
    search = await call_tool(session, "search_records", entity="care_work_queue", min_priority_score=1, limit=5)
    freshness = await call_tool(session, "get_freshness_status", entity="care_work_queue", record_id="p4")
    print("Record:", record)
    print("Search:", search["data"])
    print("Freshness:", freshness["data"])
    assert record["allowed"] and record["evidence_ids"]
    assert freshness["data"]["decision"] == "allow"
    return record, history

record_before, history_before = run_mcp_example(database, read_baseline)

Record: {'tool': 'get_record', 'allowed': True, 'denial_reason': None, 'data': {'patient_id': 'p4', 'patient_display_name': 'Sam Rivera', 'priority_score': 30, 'priority_reasons': ['Pending referral (+30)'], 'open_task_count': 0, 'overdue_task_count': 0, 'pending_referral_count': 1, 'latest_appointment_status': 'scheduled', 'projection_updated_at': '2026-10-07T07:33:39.246068'}, 'evidence_ids': ['p4']}
Search: [{'patient_id': 'p2', 'patient_display_name': 'Morgan Lee', 'priority_score': 50}, {'patient_id': 'p4', 'patient_display_name': 'Sam Rivera', 'priority_score': 30}]
Freshness: {'patient_id': 'p4', 'source_record_id': 'p4', 'operation': 'c', 'source_commit_at': '2026-10-07T07:33:39.244057Z', 'context_updated_at': '2026-10-07T07:33:44.652167Z', 'retrieval_at': '2026-10-07T07:33:44.655767Z', 'age_seconds': 0.0036, 'context_age_seconds': 0.0036, 'source_age_seconds': 5.41171, 'source_to_context_lag_seconds': 5.40811, 'budget_seconds': 300, 'decision': 'allow', 'reason': 'Context age 

## (c) Change the source, then read the CDC result

The notebook schedules a referral outside MCP. The MCP server remains read-only and sees the committed projection and history.

In [4]:
correlation_id = update_referral(
    db, "ref2", status="scheduled", scheduled_for_at=datetime.now(UTC) + timedelta(days=5)
)
print("Source update:", correlation_id)

async def read_after_change(session):
    record = await call_tool(session, "get_record", entity="care_work_queue", record_id="p4")
    history = await call_tool(session, "get_change_history", entity="care_work_queue", record_id="p4")
    print("Pending referrals:", record_before["data"]["pending_referral_count"],
          "->", record["data"]["pending_referral_count"])
    print("Newest change:", history["data"][0])
    assert record_before["data"]["pending_referral_count"] == 1
    assert record["data"]["pending_referral_count"] == 0
    assert len(history["data"]) > len(history_before["data"])

run_mcp_example(database, read_after_change)

Source update: 98936427-714d-4992-966b-b8aa9054752e
Pending referrals: 1 -> 0
Newest change: {'event_id': 'b7fa67bc-3e79-459d-8069-05c0434367da', 'event_type': 'referrals.u', 'summary': 'referrals changed; priority is now 0', 'occurred_at': '2026-10-07T07:33:45.003436'}


## (d) Route questions and inspect the call log

A deterministic client selects a tool, invokes it through MCP, and records evidence and freshness.

In [5]:
client = DeterministicMCPClient()

async def route_questions(session):
    for question in (
        "What is the status right now for this patient?",
        "What changed for this patient recently?",
        "Is this current or stale?",
    ):
        await client.ask(session, question, patient_id="p4")
    assert [entry.tool for entry in client.call_log] == [
        "get_record", "get_change_history", "get_freshness_status"
    ]
    for entry in client.call_log:
        print(entry)

run_mcp_example(database, route_questions)

ToolCallLog(question='What is the status right now for this patient?', tool='get_record', arguments={'entity': 'care_work_queue', 'record_id': 'p4'}, allowed=True, evidence_ids=['p4'], freshness=None)
ToolCallLog(question='What changed for this patient recently?', tool='get_change_history', arguments={'entity': 'care_work_queue', 'record_id': 'p4'}, allowed=True, evidence_ids=['b7fa67bc-3e79-459d-8069-05c0434367da', '4511306c-5121-4113-8ad8-830341f1914d'], freshness=None)
ToolCallLog(question='Is this current or stale?', tool='get_freshness_status', arguments={'entity': 'care_work_queue', 'record_id': 'p4', 'budget_seconds': 300}, allowed=True, evidence_ids=['p4'], freshness={'patient_id': 'p4', 'source_record_id': 'ref2', 'operation': 'u', 'source_commit_at': '2026-10-07T07:33:44.997439Z', 'context_updated_at': '2026-10-07T07:33:50.286764Z', 'retrieval_at': '2026-10-07T07:33:50.289791Z', 'age_seconds': 0.003027, 'context_age_seconds': 0.003027, 'source_age_seconds': 5.292352, 'source_

## (e) Verify denials

Entity and field restrictions return `allowed=False`. Unregistered write tools return an MCP error.

In [6]:
async def verify_denials(session):
    for arguments in (
        {"entity": "patients", "record_id": "p4"},
        {"entity": "care_work_queue", "record_id": "p4", "fields": ["birth_year"]},
    ):
        denied = await call_tool(session, "get_record", **arguments)
        print(denied["denial_reason"])
        assert not denied["allowed"]

    denied_write = await session.call_tool("update_record", {"entity": "care_work_queue"})
    print("Unknown write tool:", denied_write.isError)
    assert denied_write.isError

scenarios = [
    "A single fixed report page that always shows the same five KPIs for leadership.",
    "A nightly analytics job computing historical trend lines across all patients.",
    "An assistant that answers ad-hoc questions about a specific patient's current and past status, with per-answer freshness guarantees.",
]
for scenario in scenarios:
    print(f"{scenario}\n  -> {recommend_approach(scenario)}\n")


Entity 'patients' is not exposed by this gateway; agents do not query the production/source store directly.
Field(s) ['birth_year'] are not approved for entity 'care_work_queue'.
Unknown write tool: True


## (i) Optional: let a model choose the tools

Everything above routed questions with fixed rules. This optional cell gives the same gateway tools to a LangChain agent so a model decides which to call. The allowlist and field checks still apply, so denials come from the gateway, not the model.

Set `LLM_PROVIDER` (`gemini` or `openai`) and the matching key in `exercises/shared/.env` (see `.env.example`). Without a key this cell skips itself and the lesson stays offline.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(_SHARED / ".env")
provider = os.getenv("LLM_PROVIDER", "gemini").lower()
api_key = (
    os.getenv("OPENAI_API_KEY")
    if provider == "openai"
    else os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
)

if not api_key:
    print(f"No API key for LLM_PROVIDER={provider}; skipping the optional model-driven agent.")
else:
    from langchain.agents import create_agent
    from langchain_core.tools import StructuredTool

    if provider == "openai":
        from langchain_openai import ChatOpenAI

        model = ChatOpenAI(
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            api_key=api_key,
            base_url=os.getenv("OPENAI_BASE_URL") or None,  # set only for Microsoft Foundry
            temperature=0,
            max_tokens=512,
            timeout=120,
            max_retries=2,
        )
    else:
        from langchain_google_genai import ChatGoogleGenerativeAI

        model = ChatGoogleGenerativeAI(
            model=os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite"),
            google_api_key=api_key,
            temperature=0,
            max_output_tokens=512,
            timeout=120,
            max_retries=2,
        )

    def make_tool(name, schema):
        def run(**arguments):
            return gateway.invoke_tool(db, name, arguments).as_dict()

        return StructuredTool.from_function(
            func=run, name=name, description=schema["description"], args_schema=schema["parameters"]
        )

    agent = create_agent(
        model=model,
        tools=[make_tool(name, schema) for name, schema in gateway.TOOL_SCHEMAS.items()],
        system_prompt="Answer using only the gateway tools. Cite evidence IDs. If a tool call is denied, say so and give the reason.",
    )
    for question in [
        "What is the current status for patient p4?",
        "What is patient p4's birth_year?",
        "Delete the care_work_queue record for p4.",
    ]:
        reply = agent.invoke({"messages": [{"role": "user", "content": question}]}, config={"recursion_limit": 8})
        print(f"Q: {question}\nA: {reply['messages'][-1].content}\n")


**Recap:** You discovered tool schemas, made bounded calls, observed CDC-derived updates, inspected routing and audit logs, and confirmed explicit denials.


In [7]:
db.close()
db.bind.dispose()
database.unlink(missing_ok=True)
print("Exercise database removed; all MCP subprocesses have exited.")

Exercise database removed; all MCP subprocesses have exited.
